### Deps

```shell
pip install oracledb langchain-oracledb langchain-core

```

In [1]:
import os
import oracledb
from langchain_core.documents import Document
from langchain_oracledb import OracleVS
from langchain_oracledb.embeddings import OracleEmbeddings
from langchain_oracledb.vectorstores import DistanceStrategy
from langchain_oracledb.vectorstores import oraclevs
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASS"],
    dsn=os.environ["ORACLE_TLS_DNS"]
)

In [3]:
embedding_model = OllamaEmbeddings(model="nomic-embed-text")

In [4]:
documents = [
    Document(
        page_content="""
        1. Infrastructure for Streaming, Transcoding, & ProcessingIf you are building an app or a service that needs to store video/audio, transcode it into multiple formats, and stream it globally at scale with zero buffering.Cloudflare Stream:Top Feature: An all-in-one pipeline that handles upload, automatic multi-bitrate encoding, and global delivery via a unified API.Best For: Developers who want simple, predictable video billing without managing individual transcoders or CDNs. [1] (https://www.cloudflare.com/products/stream/)AWS Media Services:Top Feature: A massive ecosystem of specialized tools like AWS Elemental MediaLive and Amazon S3 + CloudFront.Best For: Enterprise-grade platforms requiring granular control over live broadcasts, DRM encryption, and pay-as-you-go scaling. [1] (https://aws.amazon.com/media-services/), [2] (https://aws.amazon.com/what-is/cloud-hosting/), [3] (https://www.cloudthat.com/consulting/services/media-services/)Akamai Media Solutions:Top Feature: Combines cloud compute, storage, and one of the world's largest edge delivery networks.Best For: Massive scale Live/VOD streaming that needs to minimize egress fees and monitor real-time viewer analytics. [1] (https://www.akamai.com/solutions/media)2. Offloading Website & CMS Media LibrariesIf you run a heavy website (like WordPress or WooCommerce) and want to move your heavy images and videos off your web server to make your site load faster.Next3 Offload:Top Feature: Seamlessly offloads your WordPress media files to 20+ S3-compatible cloud storage providers like Amazon S3 or Cloudflare R2.Best For: WordPress site owners looking for an automated plugin to sync files and rewrite URLs instantly. [1] (https://next3offload.com/blog/best-media-cloud-alternatives-for-wordpress/)Infinite Uploads:Top Feature: Offers unlimited sites and uploads within fixed plan structures, alongside built-in video encoding and an integrated global CDN.Best For: Agencies managing multiple client sites with massive media libraries. [1] (https://www.youtube.com/watch?v=NApX43Juhhc&vl=en&t=358)3. Collaborative Video Production & Video EditingIf you are a remote production team, agency, or editor handling tens of terabytes of raw video files and looking for the best way to edit together.The Local + Remote Hybrid (Community Consensus):Top Feature: Storing high-resolution footage on local server hardware (or a rented server rack at a colocation data center) and allowing editors to dial in using ultra-low latency remote desktop software.Best For: Video editing teams handling 15+ TB of data where traditional cloud download speeds are too slow or egress fees are too expensive.Tools to look at: Parsec or Jump Desktop for connecting to high-powered remote workstations
        """,
        metadata={
            "source": "media_hosting_cloud.txt"
        }
    ),
    Document(
        page_content="""
        Securing media streaming in the cloud requires a multi-layered architecture designed to protect video files from being intercepted, stolen, or shared. Relying on basic website passwords is not enough; modern cloud infrastructure secures content at the network edge, during transit, and right down to the viewer's device hardware.🔎 The 4 Core Layers of Cloud Streaming Security1. Gatekeeping: Token Authentication & Signed URLsBefore a user can ever access a video file, they must prove they have the right to view it.Secure Token Authentication: When a user hits "Play", your cloud backend uses a private signing key to instantly generate a short-lived token.The Payload: This token embeds parameters like the user's specific IP address, device type, and a strict expiration window (often valid for only a few minutes).Edge Validation: A global Content Delivery Network (CDN) intercepts the request and cryptographically verifies the token. If a user tries to share the raw link with a friend, the CDN drops the connection because the friend's IP won't match, or the token will have expired.2. Data in Transit: Segmented Stream EncryptionIf an attacker tries to intercept data as it moves across the web, encryption renders it unreadable.AES-128 Encryption: Modern streaming splits video into tiny, sequential file fragments (using HLS or DASH formats). Each individual chunk is encrypted using standard AES-128 encryption.CMAF Unified Packaging: Using the Common Media Application Format (CMAF) with common encryption allows a single, universally encrypted video file to look exactly the same to all devices, preventing the need to encode multiple versions of the same file to suit different players.3. Hardware Enforcement: Multi-DRM EcosystemsEncryption locks the video file, but Digital Rights Management (DRM) securely controls the keys required to unlock it. DRM interfaces directly with a device's built-in operating system or browser Content Decryption Module (CDM) so keys are never exposed in plaintext memory.A proper cloud architecture requires a Multi-DRM Setup to target different ecosystems natively:DRM EcosystemNative EnvironmentsMain StrengthsGoogle WidevineAndroid, Chrome, Firefox, ChromecastOpen-source foundation; flexible security levels (L1 hardware vs L3 software).Apple FairPlayiOS, macOS, tvOS, SafariStrict hardware enforcement; mandatory for serving video to Apple platforms.Microsoft PlayReadyWindows, Edge, Xbox, Smart TVsEnterprise standard; excellent for home entertainment consoles and 4K TVs.When a player hits an encrypted chunk, it reaches out to a cloud-based DRM License Server. The server checks the platform's custom rules (e.g., geofencing restrictions, subscription tier, concurrent stream caps) before sending an encrypted license back to allow local hardware decoding.4. Post-Leaked Deterrents: Forensic WatermarkingIf a bad actor uses external capture cards or films their screen with a camera, technical encryption stops working.Dynamic Overlays: Places a visible, moving overlay of the user's ID or IP address directly over the video frame to deter screen recording.Forensic Watermarking: Implants a completely invisible, unique mathematical signature into the video frames during cloud packaging. If a leaked stream surfaces online, scanning companies can analyze the clip, extract the invisible watermark, and trace it back to the exact subscriber account that initiated the leak.💡 Architecture Tip: For optimal implementation, host your decryption keys and DRM certificates in secure, dedicated cloud storage like an encrypted vault or a Cloud Hardware Security Module (HSM). Restrict configuration access tightly using role-based access controls (RBAC).
        """,
        metadata={
            "source": "media_security_cloud.txt"
        }
    )
]

In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 16


In [6]:
vector_store = OracleVS.from_documents(
    chunks,
    embedding_model,
    client=connection,
    table_name="SOLUTION_ARCHITECT_VECTORS",
    distance_strategy=DistanceStrategy.COSINE
)

In [7]:
oraclevs.create_index(
    connection,
    vector_store,
    params={
        "idx_name": "SOLUTION_ARCHITECT_HNSW_IDX",
        "idx_type": "HNSW",
        "neighbors": 32,
        "efConstruction": 200,
        "accuracy": 95
    }
)

print("HNSW index created.")

HNSW index created.


In [10]:
query = "What is DRM?"

results = vector_store.similarity_search(
    query,
    k=2
)

In [11]:
for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Content:", result.page_content)
    print("Metadata:", result.metadata)


Result 1
Content: (DRM) securely controls the keys required to unlock it. DRM interfaces directly with a device's built-in operating system or browser Content Decryption Module (CDM) so keys are never exposed in plaintext memory.A proper cloud architecture requires a Multi-DRM Setup to target different ecosystems natively:DRM EcosystemNative EnvironmentsMain StrengthsGoogle WidevineAndroid, Chrome, Firefox, ChromecastOpen-source foundation; flexible security levels (L1 hardware vs L3 software).Apple FairPlayiOS,
Metadata: {'source': 'media_security_cloud.txt'}

Result 2
Content: (L1 hardware vs L3 software).Apple FairPlayiOS, macOS, tvOS, SafariStrict hardware enforcement; mandatory for serving video to Apple platforms.Microsoft PlayReadyWindows, Edge, Xbox, Smart TVsEnterprise standard; excellent for home entertainment consoles and 4K TVs.When a player hits an encrypted chunk, it reaches out to a cloud-based DRM License Server. The server checks the platform's custom rules (e.g., geo